In [154]:
from model import GestureTransformer
import os
from pathlib import Path
import pandas as pd
import mediapipe as mp
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [155]:
import numpy as np
import matplotlib.pyplot as plt

# Copy-paste from MediaPipe Hands
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index finger
    (0, 9), (9, 10), (10, 11), (11, 12), # Middle finger
    (0, 13), (13, 14), (14, 15), (15, 16), # Ring finger
    (0, 17), (17, 18), (18, 19), (19, 20)  # Pinky
]

def draw_landmarks_from_features(features):
    """
    Draw right & left hand skeleton from the 84-dim feature vector.
    """
    # Extract coordinates
    rx = features[0:21]
    ry = features[21:42]
    lx = features[42:63]
    ly = features[63:84]

    plt.figure(figsize=(5, 5))
    ax = plt.gca()

    # Draw Right Hand
    if not np.allclose(rx, 0):
        for a, b in HAND_CONNECTIONS:
            plt.plot([rx[a], rx[b]], [ry[a], ry[b]], linewidth=2)
        plt.scatter(rx, ry, s=15)

    # Draw Left Hand
    if not np.allclose(lx, 0):
        for a, b in HAND_CONNECTIONS:
            plt.plot([lx[a], lx[b]], [ly[a], ly[b]], linewidth=2)
        plt.scatter(lx, ly, s=15)

    # Flip y-axis (Mediapipe uses top-left origin)
    plt.gca().invert_yaxis()

    plt.title("Hand Landmarks (from feature vector)")
    plt.axis("equal")
    plt.show()


In [156]:
import numpy as np

def landmarks_to_features(detection_results):
    """
    Generate a feature vector from both hands for a single frame.

    Feature layout (length 84):
        - Right hand x-coordinates: 0-20
        - Right hand y-coordinates: 21-41
        - Left hand x-coordinates: 42-62
        - Left hand y-coordinates: 63-83

    If a hand is missing, its entries remain zeros.

    Args:
        detection_results: MediaPipe Hands results object from hands.process()

    Returns:
        np.ndarray: Feature vector of shape (84,)
    """
    features = np.zeros(84, dtype=np.float32)

    if not detection_results.multi_hand_landmarks:
        return features

    for hand_landmarks, hand_handedness in zip(
        detection_results.multi_hand_landmarks,
        detection_results.multi_handedness
    ):
        # Extract x and y coordinates
        x_coords = [lm.x for lm in hand_landmarks.landmark]
        y_coords = [lm.y for lm in hand_landmarks.landmark]

        # Assign coordinates to proper section
        if hand_handedness.classification[0].label == "Right":
            features[0:21] = x_coords
            features[21:42] = y_coords
        else:  # Left hand
            features[42:63] = x_coords
            features[63:84] = y_coords

    return features


In [157]:
def preprocess_image(image_path: os.PathLike, hands_model=None):
    """Preprocess the image for hand tracking and return landmark features.

    Args:
        image_path (str): The path to the image file.
        frame (int), optional: If debig is set loggs files with frame number
        debug (bool, optional): Whether to display debug info. Defaults to False.

    Raises:
        ValueError: If the image cannot be loaded.

    Returns:
        np.ndarray: The processed feature vector of size (84,).
    """
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Image not found at {image_path}")
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    results = hands_model.process(image_rgb)
    
    # mp_drawing = mp.solutions.drawing_utils
    # mp_drawing_styles = mp.solutions.drawing_styles
    # mp_hands = mp.solutions.hands
    # annotated_image = image.copy()
    # if results.multi_hand_landmarks:
    #     for hand_landmarks in results.multi_hand_landmarks:
    #         mp_drawing.draw_landmarks(
    #             annotated_image,
    #             hand_landmarks,
    #             mp_hands.HAND_CONNECTIONS,
    #             mp_drawing_styles.get_default_hand_landmarks_style(),
    #             mp_drawing_styles.get_default_hand_connections_style()
    #         )
    # cv2.imwrite("./annotated_image.png", annotated_image)

    return landmarks_to_features(results)

In [158]:
def preprocess_asl_dataset(
    dataset_path: str | os.PathLike,
    final_path: str | os.PathLike
) -> None:
    """Preprocesses ASL dataset

    Args:
        dataset_path (str | os.PathLike): Path to raw files
        final_path (str | os.PathLike): Path to preprocessed files
    """
    # Splitting ratios
    TRAIN_RATIO = 0.7
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    
    # Extracting labels (folder names)
    labels = [p.name for p in Path(dataset_path).iterdir() if p.is_dir()]
    
    # Saving labels to id mapping to labels.csv
    lab2id = pd.DataFrame(
        [{"label": label, "id": idx} for idx, label in enumerate(labels)]
    )
    lab2id.to_csv(os.path.join(final_path, "labels.csv"), index=False)
    lab2id = dict(zip(lab2id["label"], lab2id["id"]))

    # Creating mediapipe model for preprocessing
    mp_hands = mp.solutions.hands
    hands_model = mp_hands.Hands(static_image_mode=True, max_num_hands=2)
    
    # Preprocessiing images
    train_df = pd.DataFrame(columns=["features", "label"])
    val_df = pd.DataFrame(columns=["features", "label"])
    test_df = pd.DataFrame(columns=["features", "label"])
    for label in labels:
        img_paths = [
            os.path.join(dataset_path, f"{label}/{i}.jpg")
            for i in range(10)
        ]

        # First split: train vs temp (val + test)
        train_paths, temp_paths = train_test_split(
            img_paths,
            test_size=VAL_RATIO + TEST_RATIO,
            shuffle=True,
            random_state=42
        )

        # Second split: val vs test
        val_paths, test_paths = train_test_split(
            temp_paths,
            test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
            shuffle=True,
            random_state=42
        )

        # Preprocess images and store results
        for path in tqdm(train_paths, desc=f"Preprocessing Label {label} Train"):
            # TODO Add data augmentations
            features = preprocess_image(path, hands_model)
            train_df.loc[len(train_df)] = ({"features": features, "label": lab2id[label]})

        for path in tqdm(val_paths, desc=f"Preprocessing Label {label} Val"):
            features = preprocess_image(path, hands_model)
            val_df.loc[len(val_df)] = ({"features": features, "label": lab2id[label]})

        for path in tqdm(test_paths, desc=f"Preprocessing Label {label} Test"):
            features = preprocess_image(path, hands_model)
            test_df.loc[len(test_df)] = ({"features": features, "label": lab2id[label]})

    test_df.to_parquet(os.path.join(final_path, "test.parquet"), index=False)
    train_df.to_parquet(os.path.join(final_path, "train.parquet"), index=False)
    val_df.to_parquet(os.path.join(final_path, "val.parquet"), index=False)


In [159]:
preprocess_asl_dataset(
    dataset_path="../../data/raw/data",
    final_path="../../data/processed"
)

Preprocessing Label Z Test: 100%|██████████| 2/2 [00:00<00:00, 26.76it/s]


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.